# Vertex Asset History Enrichment

Run this notebook in ArcGIS Pro 3.7.2. It reads `Vertex_Assets.gdb` and creates only `Vertex_Assets_Enriched.gdb`; it never updates or deletes the source.

## 1. Define Requirements and Inputs

The output retains the minimal Indoors hierarchy and existing generated assets, then adds exact lifecycle cohorts, bounded inspection history, repair history, and completed/active work orders. Narratives are deterministic by default. Optionally set the localhost LLM values below; invalid, unavailable, or over-budget responses use deterministic fallback text, and the manifest records accepted and fallback counts.

In [ ]:
from datetime import date
import importlib.util
import json
from pathlib import Path
import sys

import arcpy

REPOSITORY = Path(r"c:\dev\sandbox\demo-data-generator")
SOURCE_GDB = Path(r"c:\Workspace\Sandbox\data\ILM_asset_gen\Medical Indoors_3141f0\GDBs\Vertex_Assets.gdb")
OUTPUT_GDB = SOURCE_GDB.with_name("Vertex_Assets_Enriched.gdb")
MANIFEST_PATH = OUTPUT_GDB.with_name("Vertex_Assets_Enriched_manifest.json")
REFERENCE_DATE = date(2026, 9, 28)
ALLOW_OVERWRITE = False
LLM_ENDPOINT = None  # Example: "http://localhost:11434/api/generate"
LLM_MODEL = None  # Example: "mistral"
LLM_MAX_REQUESTS = 100
LLM_TIMEOUT_SECONDS = 120

sys.path.insert(0, str(REPOSITORY / "src"))
spec = importlib.util.spec_from_file_location(
    "vertex_enrichment", REPOSITORY / "scripts" / "enrich_vertex_asset_history.py"
)
if spec is None or spec.loader is None:
    raise ImportError("Unable to load the Vertex enrichment script.")
vertex_enrichment = importlib.util.module_from_spec(spec)
sys.modules["vertex_enrichment"] = vertex_enrichment
spec.loader.exec_module(vertex_enrichment)

assert SOURCE_GDB != OUTPUT_GDB
assert arcpy.Exists(str(SOURCE_GDB))
assert not LLM_ENDPOINT or LLM_MODEL
print("ArcGIS Pro:", arcpy.GetInstallInfo()["Version"])
print("Source:", SOURCE_GDB)
print("Output:", OUTPUT_GDB)
print("Overwrite enabled:", ALLOW_OVERWRITE)

## 2. Define the Guarded Workflow

Create the output, remap GlobalID relationships, generate lifecycle history, validate the result, and write the manifest inside one cleanup boundary. Existing output is preserved unless overwrite is explicitly enabled.

In [ ]:
def run_vertex_enrichment():
    output = OUTPUT_GDB.resolve()
    if output.exists() or arcpy.Exists(str(output)):
        if not ALLOW_OVERWRITE:
            raise FileExistsError(
                f"Output exists: {output}. Set ALLOW_OVERWRITE = True to replace it."
            )
        vertex_enrichment._delete_output(output)
    MANIFEST_PATH.unlink(missing_ok=True)

    config = vertex_enrichment.EnrichmentConfig(
        reference_date=REFERENCE_DATE,
        inspection_start=date(2021, 9, 28),
        inspection_end=date(2026, 3, 28),
        seed=3141,
    )
    narrator = (
        vertex_enrichment.LocalJsonNarrator(
            LLM_ENDPOINT,
            LLM_MODEL,
            timeout_seconds=LLM_TIMEOUT_SECONDS,
            max_requests=LLM_MAX_REQUESTS,
        )
        if LLM_ENDPOINT
        else None
    )
    source_schema = vertex_enrichment._schema(str(SOURCE_GDB))
    vertex_enrichment._ensure_empty_source_children(source_schema)

    try:
        vertex_enrichment._create_minimal_output(str(SOURCE_GDB), str(output))
        output_source = vertex_enrichment.validate_indoors_source(arcpy, str(output))
        output_schema = vertex_enrichment.create_asset_management_schema(
            arcpy, str(output), output_source
        )
        for field_name, field_type, length in vertex_enrichment.LIFECYCLE_EXTENSION_FIELDS:
            vertex_enrichment._add_field(
                output_schema.asset_lifecycle, field_name, field_type, length
            )

        vertex_enrichment._copy_asset_schema_rows(
            source_schema, output_schema, str(output)
        )
        assets = vertex_enrichment._asset_seeds(output_schema)
        _, facility_names = vertex_enrichment._hierarchy(str(output))
        plan = vertex_enrichment.build_enrichment_plan(
            assets,
            config,
            narrator=narrator,
            facility_name=next(iter(facility_names.values())),
        )
        vertex_enrichment._apply_plan(output_schema, plan)
        validation = vertex_enrichment._validate_output(
            str(SOURCE_GDB), str(output), output_schema, config
        )
        manifest = {
            "created_utc": vertex_enrichment.DateTime.now(
                vertex_enrichment.timezone.utc
            ).isoformat(timespec="seconds"),
            "source_geodatabase": str(SOURCE_GDB),
            "output_geodatabase": str(output),
            "source_modified": False,
            "reference_date": config.reference_date.isoformat(),
            "inspection_window": {
                "start": config.inspection_start.isoformat(),
                "end": config.inspection_end.isoformat(),
            },
            "seed": config.seed,
            "narrative_generation": vertex_enrichment.narrative_run_metadata(narrator),
            "validation": validation,
        }
        MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        return manifest
    except Exception:
        vertex_enrichment._delete_output(output)
        MANIFEST_PATH.unlink(missing_ok=True)
        raise
    finally:
        arcpy.management.ClearWorkspaceCache()


print("Guarded Vertex workflow ready.")

## 3. Run the Workflow

Generate the separate Vertex output. Any exception removes the partial output and manifest while leaving the source geodatabase unchanged.

In [ ]:
manifest = run_vertex_enrichment()
validation = manifest["validation"]
print("Validated output:", OUTPUT_GDB)
print("Manifest:", MANIFEST_PATH)
print(json.dumps(manifest["narrative_generation"], indent=2))

## 4. Validate the Output

Check exact lifecycle cohorts, repair and work-order counts, orphan references, warranty totals, and risk totals.

In [ ]:
assert validation["lifecycle_assets"] == 1003
assert validation["lifecycle_stages"] == {
    "Recently Installed": 201,
    "Retrofit/Replacement": 301,
    "Within Service Life": 501,
}
assert validation["repairs"] == 271
assert validation["historical_work_orders"] == 451
assert validation["active_work_orders"] == 50
assert validation["orphan_references"] == 0
assert sum(validation["warranty_statuses"].values()) == 1003
assert sum(validation["risk_levels"].values()) == 1003
print(json.dumps(validation, indent=2))

## 5. Verify Source Preservation and Telemetry

Confirm the source child tables remain empty, the manifest exists, and narrative counters satisfy their accounting invariants.

In [ ]:
source_child_counts = {
    name: vertex_enrichment._row_count(str(SOURCE_GDB / name))
    for name in vertex_enrichment.CHILD_TABLES
}
assert not any(source_child_counts.values())
assert manifest["source_modified"] is False
assert MANIFEST_PATH.exists()

statistics = manifest["narrative_generation"]["statistics"]
assert statistics["total_calls"] == (
    statistics["accepted_responses"] + statistics["fallback_responses"]
)
assert statistics["fallback_responses"] == (
    statistics["budget_fallbacks"]
    + statistics["invalid_response_fallbacks"]
    + statistics["request_error_fallbacks"]
)
print("Source child tables remain empty:", source_child_counts)
print("Notebook validation completed.")